# BRISC — DRL: Brain Tumour MRI

**Set `AGENT_NAME` in Cell 1 to pick the algorithm. Everything else is automatic.**

| Variable | Options |
|---|---|
| `AGENT_NAME` | `'DQN'` · `'DuelingDDQN'` · `'DDPG'` |

**Paradigm:** DQN and DuelingDDQN are **boundary tracers** (8 directional actions on a
64×64 patch — the trajectory IS the segmentation). DDPG is the **continuous baseline**
(mask morphology). Reward for the continuous path is `iou_delta` (smoother for small
targets); the tracing path uses the −distance-to-boundary reward.

**BRISC baseline:** val Dice 0.8290 · test Dice 0.8351 · HD95 8.36 px
**Key EDA facts:** 9 586 pairs · variable sizes 212–624 px (resized to 256) · mean tumour area 1.74%
· mostly single blob (mean 1.1 components) · HD95 per-patient range 1.4 → 78 px (high upside for DRL)

> Run §3 (dry-run) first, then §4 (full training).


## 0 · Install + locate package

In [ ]:
import subprocess, sys
from pathlib import Path

init_files = list(Path('/kaggle/input').rglob('iteris/__init__.py'))
if not init_files:
    raise RuntimeError('iteris-pkg not attached. Add it in the right-hand panel → Datasets.')
PKG_ROOT = init_files[0].parent.parent
subprocess.run(['pip', 'install', '-r', str(PKG_ROOT / 'requirements.txt'),
                '--quiet', '--upgrade'], check=True)
sys.path.insert(0, str(PKG_ROOT))
print(f'✓ Package at: {PKG_ROOT}')

## 1 · Configure — set algorithm here

In [ ]:
# ══════════════════════════════════════════════════════════════════════
AGENT_NAME = 'DQN'   # DQN | DuelingDDQN | DDPG  (DQN, DuelingDDQN = 14-action local mask refinement; DDPG = continuous baseline)
# ══════════════════════════════════════════════════════════════════════

from iteris.config import load_drl_class_config, resolve_agent_config, load_config
from iteris.utils  import get_device

cfg_full     = load_drl_class_config(str(PKG_ROOT / 'configs' / 'brisc_drl_tumor.yaml'))
cfg          = resolve_agent_config(cfg_full, AGENT_NAME)
baseline_cfg = load_config(str(PKG_ROOT / 'configs' / cfg['baseline_cfg_name']))

cfg['checkpoint_dir'] = '/kaggle/working'

# Auto-detect BRISC data root and U-Net checkpoint
brisc_roots = [p for p in Path('/kaggle/input').rglob('brisc2025') if p.is_dir()]
if not brisc_roots:
    brisc_roots = [p for p in Path('/kaggle/input').rglob('BRISC') if p.is_dir()]
if brisc_roots:
    baseline_cfg['data_root'] = str(brisc_roots[0])
else:
    print('[WARN] BRISC data root not found — attach the dataset in the right panel')

ckpt_cands = [p for p in Path('/kaggle/input').rglob('brisc_best.pt')]
if ckpt_cands:
    cfg['baseline_checkpoint'] = str(ckpt_cands[0])
else:
    print('[WARN] brisc_best.pt not found — attach the baseline outputs dataset')

print(f'Agent           : {cfg["agent_type"]}')
print(f'Target class    : {cfg["target_class"]} ({cfg["class_name"]})')
print(f'Reward mode     : {cfg["reward_mode"]}  (IoU-delta — smoother for small targets)')
print(f'BRISC data root : {baseline_cfg["data_root"]}')
print(f'Baseline ckpt   : {cfg["baseline_checkpoint"]}')
print(f'Train steps     : {cfg["train_steps"]}')
print(f'Buffer size     : {cfg["buffer_size"]}')
print(f'shift_px        : {cfg["shift_px"]}  (1 px — precision mode for small tumours)')
print(f'sdt_clip        : {cfg["sdt_clip"]}  (tighter SDT range for small tumours)')
print(f'Device          : {get_device()}')

## 2 · Warm-start — pre-compute U-Net init masks

Runs the U-Net over all splits once and caches `{image, gt_mask, init_mask}` dicts.  
The DRL training loop never touches the U-Net again — avoids repeated backbone inference  
during RL steps.

**BRISC notes:**
- Images are resized from variable sizes (up to 624 px) to 256×256 by the transforms.
- z-score normalisation is applied (MRI standard — robust to scanner drift).
- `min_area_fraction=0.005` is looser than CAMUS (0.01) to avoid dropping genuinely small tumours.
- Expected drop rate is very low — EDA showed 0 samples with area < 0.5%.

~8–15 min on T4 (9 586 samples total).

In [ ]:
from iteris.warm_start import precompute_init_masks
import numpy as np

train_samples, val_samples, test_samples = precompute_init_masks(
    baseline_cfg        = baseline_cfg,
    baseline_checkpoint = cfg['baseline_checkpoint'],
    target_class        = cfg['target_class'],
    min_area_fraction   = cfg.get('min_area_fraction', 0.005),
    tumor_type_filter   = cfg.get('tumor_type_filter', None),
    # tumor_type_filter='glioma' | 'meningioma' | 'pituitary' | None (all)
    # Set in the DRL config (e.g. brisc_drl_glioma.yaml) — None keeps all
    # tumor types (the original binary tumor/non-tumor task used in the
    # first BRISC runs).
)

print(f'\nSamples — train: {len(train_samples)}  val: {len(val_samples)}  test: {len(test_samples)}')
if cfg.get('tumor_type_filter'):
    print(f'  (filtered to tumor_type = {cfg["tumor_type_filter"]!r})')

def dice_to_iou(d): return d / (2.0 - d + 1e-8)

init_dices = [
    float((s['init_mask'] & s['gt_mask']).sum() * 2 /
          (s['init_mask'].sum() + s['gt_mask'].sum() + 1e-6))
    for s in val_samples
]
init_ious = [dice_to_iou(d) for d in init_dices]
gt_areas  = [s['gt_mask'].mean() * 100 for s in val_samples]

print(f'U-Net init Dice on val (tumor): mean {np.mean(init_dices):.4f}  '
      f'median {np.median(init_dices):.4f}')
print(f'U-Net init IoU  on val (tumor): mean {np.mean(init_ious):.4f}  '
      f'median {np.median(init_ious):.4f}')
print(f'GT tumour area  (val)         : mean {np.mean(gt_areas):.2f}%  '
      f'range [{min(gt_areas):.2f}%, {max(gt_areas):.2f}%]')


## 3 · OPTIONAL — Dry-run sanity check (~3 min)

Runs 600 training steps on 30 train / 10 val samples.  
Self-contained `deepcopy(cfg)` — does NOT mutate `cfg`, `train_samples`, or `val_samples`.  
Best Dice at 600 steps is meaningless; what matters is that it runs without error.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
RUN_DRY_RUN = False   # Set True for a quick sanity check before full training
# ══════════════════════════════════════════════════════════════════════

if RUN_DRY_RUN:
    import copy
    from iteris.drl_training import run_drl_training

    _dry = copy.deepcopy(cfg)
    _dry.update(dict(
        train_steps         = 600,
        prefill_steps       = 500,
        buffer_size         = 3000,
        eval_every          = 200,
        epsilon_decay_steps = 400,
        batch_size          = 32,
    ))
    _dry_result = run_drl_training(_dry, train_samples[:30], val_samples[:10])
    print(f'\n✓ Dry run passed. Best val score (meaningless at 600 steps): {_dry_result["best_dice"]:.4f}')
    print(f'  cfg["train_steps"] still = {cfg["train_steps"]}  (unchanged)')
    print('  → Safe to run §4 below for the real training run.')
else:
    print('Dry run skipped (RUN_DRY_RUN = False). Proceed directly to §4.')


## 4 · Full training

| Agent | Paradigm | Est. time on T4 | Steps |
|---|---|---|---|
| DQN / DuelingDDQN | Boundary tracing | ~1.5–2 hr | 25 000 |
| DDPG | Continuous baseline | ~5–6 hr | 100 000 |

> BRISC has ~7 500 training samples (vs 1 500 for CAMUS per class). For tracing the SDT
> cache is not used (the env precomputes a GT-boundary distance field at reset instead);
> for DDPG `cache_sdt=True` is critical for throughput.

Checkpoint saved whenever the val score improves:
- Tracing: `/kaggle/working/brisc_<agent>_trace_c1_best.pt`
- DDPG:    `/kaggle/working/brisc_<agent>_tumor_best.pt`


In [ ]:
from iteris.drl_training import run_drl_training

result    = run_drl_training(cfg, train_samples, val_samples)
agent     = result['agent']
history   = result['history']
best_dice = result['best_dice']

IS_TRACING = cfg.get('env_class') == 'contour_tracing'
if IS_TRACING:
    print(f'\n✓ Training complete. Best val final-Dice (tumor): {best_dice:.4f}')
else:
    # iou_delta mode stores best IoU in best_dice
    print(f'\n✓ Training complete. Best val final-IoU (tumor): {best_dice:.4f}')
    print(f'  Equiv. Dice ≈ {2*best_dice/(1+best_dice):.4f}')
print(f'  Checkpoint : {result["checkpoint"]}')


## 5 · Visualisation setup

Defines `replay_episode()` and `ENV_KW`. Used by all cells below.  

**BRISC-specific ENV_KW differences vs CAMUS:**
- `shift_px=1` (finer — small tumours need sub-pixel precision)
- `sdt_clip=15.0` (tighter — small structures have shorter signed-distance gradients)
- `cont_morph_scale=0.15` (±2.25 px morphological shift vs ±5 px for CAMUS)
- `cont_trans_scale=0.01` (±2.56 px translation vs ±5.12 px for CAMUS)
- `reward_mode='iou_delta'` — HD95 is NOT computed during episodes (skipped for speed)

HD95 is computed separately in §6 for display purposes only (not used in training reward).

In [ ]:
from iteris.drl_training import evaluate_agent
from iteris.env           import SegmentationEnv

ENV_KW = dict(
    action_type  = cfg.get('action_type', 'discrete'),
    max_steps    = cfg.get('max_steps', 20),
    reward_mode  = cfg.get('reward_mode', 'iou_delta'),
    sdt_clip     = cfg.get('sdt_clip', 15.0),
    shift_px     = cfg.get('shift_px', 1),
    fail_thresh  = cfg.get('fail_thresh', 0.0),
    fail_n       = cfg.get('fail_n', 2),
)

DISCRETE_NAMES = SegmentationEnv.DISCRETE_NAMES   # 14 action names including smooth+stop


## 6 · Sample comparison — best / median / worst gains

Shows U-Net init → agent refined → GT for the three representative test cases.  
HD95 is computed here for display only (not used during training).

In [ ]:
if IS_TRACING:
    fig = cv.plot_trace_comparison(
        replays, baseline_cfg, cfg, CLASS_IDX, CLASS_NAME,
        out_path=f'/kaggle/working/brisc_{cfg["agent_type"].lower()}_comparison.png')
    plt.show()
else:
    picks = [('BEST gain', replays[-1]), ('MEDIAN gain', replays[N_VIZ//2]), ('WORST gain', replays[0])]
    fig, axes = plt.subplots(len(picks), 4, figsize=(18, 4.5*len(picks)))
    for row, (label, r) in enumerate(picks):
        s          = r['sample']
        final_mask = r['masks'][-1]
        img, init, gt = s['image'], s['init_mask'], s['gt_mask']
        hd_init  = hd95_px(init, gt)
        hd_final = hd95_px(final_mask, gt)
        cells = [
            ('Input (MRI)',                   None,       None,        None),
            ('U-Net init',                    init,       r['init_d'], hd_init),
            (f'{cfg["agent_type"]} refined',  final_mask, r['final_d'], hd_final),
            ('Ground Truth',                  gt,         1.0,         0.0),
        ]
        for col, (title, mask, d, h) in enumerate(cells):
            ax = axes[row, col]
            ax.imshow(img, cmap='gray')
            if mask is not None:
                ax.imshow(np.ma.masked_where(mask == 0, mask), cmap=CMAP_TUMOR, alpha=0.5)
            if d is not None and h is not None:
                iou = dice_to_iou(d)
                ax.set_title(f'{title}\nDice {d:.3f}  IoU {iou:.3f}  HD95 {h:.1f}px', fontsize=9)
            else:
                ax.set_title(f'{title}\n[{label}]  dDice {r["gain_dice"]:+.4f}  dIoU {r["gain_iou"]:+.4f}',
                             fontsize=9)
            ax.axis('off')
    plt.suptitle(f'BRISC {cfg["agent_type"]} — Tumour Refinement', fontsize=13)
    plt.tight_layout()
    out = f'/kaggle/working/brisc_{cfg["agent_type"].lower()}_comparison.png'
    plt.savefig(out, dpi=150); plt.show(); print(f'Saved to {out}')


## 7 · Iteration playback — all steps on best-gain sample

Rose overlay = agent mask · Green contour = GT boundary.  
This is the data the UI Iteration Playback page consumes.

In [ ]:
if IS_TRACING:
    fig = cv.plot_trajectory_playback(
        replays[-1], cfg, CLASS_NAME,
        out_path=f'/kaggle/working/brisc_{cfg["agent_type"].lower()}_playback.png')
    plt.show()
else:
    from scipy import ndimage
    r = replays[-1]; s = r['sample']
    gt_edge = s['gt_mask'] ^ ndimage.binary_erosion(s['gt_mask'], iterations=1)
    n_steps = len(r['masks'])
    ncols = 5; nrows = int(np.ceil(n_steps / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2*ncols, 3.2*nrows))
    axes = np.atleast_2d(axes).flatten()
    for i, (mask, dice, iou) in enumerate(zip(r['masks'], r['dices'], r['ious'])):
        ax = axes[i]
        ax.imshow(s['image'], cmap='gray')
        ax.imshow(np.ma.masked_where(mask == 0, mask), cmap=CMAP_TUMOR, alpha=0.55)
        ax.imshow(np.ma.masked_where(~gt_edge.astype(bool), gt_edge), cmap='Greens', alpha=0.8)
        if i == 0:
            title = f't=0 (init)\nDice {dice:.3f}  IoU {iou:.3f}'
        else:
            a = r['actions'][i - 1]
            if agent.action_type == 'discrete':
                a_str = DISCRETE_NAMES[int(a)]
            else:
                a_str = f'm={a[0]:+.3f} dy={a[1]*256:+.1f}px dx={a[2]*256:+.1f}px'
            title = f't={i}  {a_str}\nDice {dice:.3f}  D{dice-r["dices"][i-1]:+.4f}'
        ax.set_title(title, fontsize=7); ax.axis('off')
    for j in range(n_steps, len(axes)): axes[j].axis('off')
    plt.suptitle(f'{cfg["agent_type"]} playback — tumor  (green = GT boundary)', fontsize=12)
    plt.tight_layout()
    out = f'/kaggle/working/brisc_{cfg["agent_type"].lower()}_playback.png'
    plt.savefig(out, dpi=150); plt.show(); print(f'Saved to {out}')


## 8 · Behaviour analysis — trajectories + action distribution

**Left:** Dice trajectory per episode (thin lines = individual, thick = mean).  
**Centre:** IoU trajectory — the actual reward signal the agent trained on.  
**Right:** Action distribution (discrete) or morphological action histogram (DDPG).

What to look for:
- Mean IoU should trend upward (→ agent is improving on these test episodes)
- For discrete agents: no-op should dominate once an improvement is found (→ early-stop working)
- For DDPG: morph histogram should be slightly right-skewed (net dilate bias on small targets)

In [ ]:
if IS_TRACING:
    fig = cv.plot_direction_behaviour(
        replays, cfg, CLASS_NAME,
        out_path=f'/kaggle/working/brisc_{cfg["agent_type"].lower()}_behaviour.png')
    plt.show()
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    ax = axes[0]
    for r in replays: ax.plot(r['dices'], alpha=0.5, lw=1)
    max_len = max(len(r['dices']) for r in replays)
    padded  = np.stack([np.pad(r['dices'], (0, max_len - len(r['dices'])),
                               constant_values=r['dices'][-1]) for r in replays])
    ax.plot(padded.mean(axis=0), color='black', lw=2.5, label='mean')
    ax.set(xlabel='Step', ylabel='Dice', title='Tumor Dice trajectory')
    ax.legend(); ax.grid(alpha=0.3)
    ax = axes[1]
    iou_padded = np.stack([np.pad(r['ious'], (0, max_len - len(r['ious'])),
                                  constant_values=r['ious'][-1]) for r in replays])
    for r in replays: ax.plot(r['ious'], alpha=0.5, lw=1)
    ax.plot(iou_padded.mean(axis=0), color='black', lw=2.5, label='mean')
    ax.set(xlabel='Step', ylabel='IoU', title='Tumor IoU trajectory (reward signal)')
    ax.legend(); ax.grid(alpha=0.3)
    ax = axes[2]
    all_actions = [a for r in replays for a in r['actions']]
    if agent.action_type == 'discrete':
        counts = np.bincount(all_actions, minlength=len(DISCRETE_NAMES))
        bars = ax.bar(range(len(DISCRETE_NAMES)), counts / counts.sum())
        bars[-1].set_color('#22C55E')
        ax.set_xticks(range(len(DISCRETE_NAMES)))
        ax.set_xticklabels(DISCRETE_NAMES, rotation=30)
        ax.set(ylabel='Frequency', title='Action distribution'); ax.grid(alpha=0.3, axis='y')
    else:
        arr = np.array(all_actions)
        ax.hist(arr[:, 0], bins=30, alpha=0.7, color=CLASS_COLOR, label='morph')
        ax.axvline(0, color='k', lw=0.8, ls='--')
        ax.set(xlabel='Morph action', title='DDPG morph distribution')
        ax.legend(); ax.grid(alpha=0.3)
    plt.suptitle(f'BRISC {cfg["agent_type"]} — tumor behaviour')
    plt.tight_layout()
    out = f'/kaggle/working/brisc_{cfg["agent_type"].lower()}_behaviour.png'
    plt.savefig(out, dpi=150); plt.show()
    print(f'\n-- {cfg["agent_type"]} tumor behaviour --')
    print(f'  Init  Dice avg : {np.mean([r["init_d"]    for r in replays]):.4f}')
    print(f'  Final Dice avg : {np.mean([r["final_d"]   for r in replays]):.4f}')
    print(f'  Delta Dice avg : {np.mean([r["gain_dice"] for r in replays]):+.4f}')
    print(f'  Final IoU  avg : {np.mean([r["final_iou"] for r in replays]):.4f}')
    print(f'  Delta IoU  avg : {np.mean([r["gain_iou"]  for r in replays]):+.4f}')
    print(f'  Avg ep length  : {np.mean([len(r["dices"]) - 1 for r in replays]):.1f} steps')


## 9 · Learning curves

Left: val IoU over training — **this is the metric that drives checkpoint saving** for BRISC  
(iou_delta mode stores best IoU as `best_dice` in the checkpoint).  
Right: ΔIoU (final − init) — should trend from negative/zero → positive during training.

Healthy training signature:
- ΔIoU positive and increasing after ~10–15k steps
- Final IoU > Init IoU on val
- No persistent oscillation (ΔIoU alternating ±)

In [ ]:
if IS_TRACING:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    ax1.plot(history['step'], history['final_dice_mean'], label='Trace Dice', color=CLASS_COLOR)
    ax1.set(xlabel='Training step', ylabel='Mean Val Dice', title='Boundary-tracing learning curve — tumor')
    ax1.legend(); ax1.grid(alpha=0.3)
    if 'closure_rate' in history.columns:
        ax2.plot(history['step'], history['closure_rate'], color='C2')
        ax2.set(xlabel='Training step', ylabel='Closure rate',
                title=f'Closure rate — {cfg["agent_type"]} (tumor)')
    ax2.grid(alpha=0.3)
    plt.suptitle(f'BRISC {cfg["agent_type"]}_TRACE — tumor learning curves')
    plt.tight_layout()
    out = f'/kaggle/working/brisc_{cfg["agent_type"].lower()}_curves.png'
    plt.savefig(out, dpi=150); plt.show(); print(f'Saved to {out}')
else:
    def dice_to_iou(d): return d / (2.0 - d + 1e-8)
    init_iou_hist  = history['init_dice_mean'].apply(dice_to_iou)
    final_iou_hist = history['final_dice_mean'].apply(dice_to_iou)
    delta_iou_hist = final_iou_hist - init_iou_hist
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(history['step'], init_iou_hist,  label='Init IoU (U-Net)', ls='--', alpha=0.6)
    ax1.plot(history['step'], final_iou_hist, label=f'Final IoU ({cfg["agent_type"]})', color=CLASS_COLOR)
    ax1.set(xlabel='Training step', ylabel='Mean Val IoU', title='BRISC — Refinement curve (IoU)')
    ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(history['step'], delta_iou_hist, color=CLASS_COLOR)
    ax2.axhline(0, color='k', lw=0.8)
    ax2.fill_between(history['step'], delta_iou_hist, 0,
                     where=(delta_iou_hist > 0), alpha=0.15, color='green')
    ax2.fill_between(history['step'], delta_iou_hist, 0,
                     where=(delta_iou_hist < 0), alpha=0.15, color='red')
    ax2.set(xlabel='Training step', ylabel='Delta IoU',
            title=f'Refinement gain — {cfg["agent_type"]} (tumor)')
    ax2.grid(alpha=0.3)
    plt.suptitle(f'BRISC {cfg["agent_type"]} — tumor learning curves')
    plt.tight_layout()
    out = f'/kaggle/working/brisc_{cfg["agent_type"].lower()}_curves.png'
    plt.savefig(out, dpi=150); plt.show(); print(f'Saved to {out}')


## 10 · Test-set evaluation + summary JSON

Full test set evaluation with greedy policy.  
Computes Dice, IoU, and HD95 (HD95 computed here even though it wasn't in the training reward —  
it's important for the paper comparison vs the U-Net baseline of 8.36 px).

~7–15 min depending on test set size.

In [ ]:
import json

if IS_TRACING:
    test_metrics = cv.evaluate_trace_testset(agent, test_samples, ENV_KW)
    print(json.dumps(test_metrics, indent=2))
    summary = {**test_metrics,
               'agent_type':    cfg['agent_type'],
               'target_class':  cfg['target_class'],
               'class_name':    cfg['class_name'],
               'paradigm':      'contour_tracing',
               'baseline_dice': 0.8351,
               'baseline_hd95': 8.36}
    out = f'/kaggle/working/brisc_{cfg["agent_type"].lower()}_test_results.json'
    with open(out, 'w') as f: json.dump(summary, f, indent=2)
    print(f'\nSaved to {out}')
    print(json.dumps(summary, indent=2))
else:
    from iteris.drl_training import evaluate_agent
    test_metrics = evaluate_agent(agent, test_samples, env_kwargs=ENV_KW)
    test_hd95s = []
    print('Computing HD95 on test samples...')
    for s in test_samples:
        env  = SegmentationEnv(s['image'], s['gt_mask'], s['init_mask'], **ENV_KW)
        state = env.reset()
        while True:
            if agent.action_type == 'discrete':
                a = agent.select_action(state, epsilon=0.0)
            else:
                a = agent.select_action(state, explore=False)
            state, _, done, _ = env.step(a)
            if done:
                break
        test_hd95s.append(hd95_px(env.mask, s['gt_mask']))
    valid_hd95 = [h for h in test_hd95s if not np.isnan(h)]
    test_metrics['final_hd95_mean'] = float(np.mean(valid_hd95)) if valid_hd95 else float('nan')
    test_metrics['init_iou_mean']   = float(dice_to_iou(test_metrics['init_dice_mean']))
    test_metrics['final_iou_mean']  = float(dice_to_iou(test_metrics['final_dice_mean']))
    test_metrics['delta_iou_mean']  = float(test_metrics['final_iou_mean'] - test_metrics['init_iou_mean'])
    summary = {**test_metrics,
               'agent_type':    cfg['agent_type'],
               'target_class':  cfg['target_class'],
               'class_name':    cfg['class_name'],
               'reward_mode':   cfg['reward_mode'],
               'best_val_iou':  float(best_dice),
               'baseline_dice': 0.8351,
               'baseline_hd95': 8.36}
    out = f'/kaggle/working/brisc_{cfg["agent_type"].lower()}_test_results.json'
    with open(out, 'w') as f: json.dump(summary, f, indent=2)
    print(f'\nSaved to {out}')
    print(json.dumps(summary, indent=2))
